# Lab 01: Flow Matching Text Generator

> **Compute Note**: Full training on TinyStories takes 20-40 minutes on GPU. CPU training will be significantly slower. If using Colab, select a GPU runtime.

In this lab you will:
1. Build a flow-matching text generator from scratch.
2. Implement an SDE-based baseline for comparison.
3. Compare generation quality, NFE, and wall-clock time.

Fill in all `# TODO` sections.

In [ ]:
%pip install torch numpy matplotlib datasets --quiet

In [ ]:
import math
import time
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Task 1: Build a Simple Tokenizer

Create a word-level tokenizer with special tokens.

In [ ]:
class SimpleTokenizer:
    """Word-level tokenizer with special tokens."""

    PAD = "<pad>"
    UNK = "<unk>"
    BOS = "<bos>"
    EOS = "<eos>"

    def __init__(self, max_vocab_size: int = 5000):
        self.max_vocab_size = max_vocab_size
        self.word2idx = {}
        self.idx2word = []

    def build_vocab(self, texts: list[str]):
        """Build vocabulary from a list of text strings."""
        # TODO: Implement vocabulary building
        # 1. Add special tokens: PAD, UNK, BOS, EOS (in that order)
        # 2. Count word frequencies across all texts (split on whitespace, lowercase)
        # 3. Add the most common words up to max_vocab_size
        # 4. Populate self.word2idx and self.idx2word
        pass

    @property
    def vocab_size(self) -> int:
        return len(self.idx2word)

    @property
    def pad_id(self) -> int:
        return self.word2idx[self.PAD]

    def encode(self, text: str, max_len: int = 64) -> list[int]:
        """Encode text to token IDs with BOS/EOS and padding."""
        # TODO: Implement encoding
        # 1. Split text on whitespace and lowercase
        # 2. Convert words to IDs (use UNK for unknown words)
        # 3. Add BOS at start and EOS at end
        # 4. Truncate to max_len if needed
        # 5. Pad with PAD to max_len
        # Return list of integer IDs
        pass

    def decode(self, ids: list[int]) -> str:
        """Decode token IDs back to text."""
        # TODO: Implement decoding
        # 1. Convert IDs to words
        # 2. Stop at EOS or PAD
        # 3. Skip BOS
        # Return joined string
        pass

In [ ]:
# Load a subset of TinyStories for training
# If datasets is not available, we fall back to synthetic data
try:
    from datasets import load_dataset
    ds = load_dataset("roneneldan/TinyStories", split="train[:5000]")
    texts = [row["text"] for row in ds]
    print(f"Loaded {len(texts)} TinyStories samples")
except Exception as e:
    print(f"Could not load TinyStories: {e}")
    print("Using synthetic data instead.")
    templates = [
        "the {adj} {noun} {verb} {prep} the {noun2}",
        "once upon a time there was a {adj} {noun}",
        "the {noun} and the {noun2} were {adj} friends",
    ]
    adjs = ["big", "small", "happy", "sad", "red", "blue", "little", "old"]
    nouns = ["cat", "dog", "bird", "fish", "bear", "rabbit", "fox", "mouse"]
    verbs = ["sat", "ran", "jumped", "played", "slept", "walked"]
    preps = ["on", "in", "over", "under", "near", "by"]
    texts = []
    for _ in range(2000):
        t = np.random.choice(templates)
        text = t.format(
            adj=np.random.choice(adjs),
            noun=np.random.choice(nouns),
            noun2=np.random.choice(nouns),
            verb=np.random.choice(verbs),
            prep=np.random.choice(preps),
        )
        texts.append(text)

print(f"Sample text: {texts[0][:100]}")

In [ ]:
# Build tokenizer
tokenizer = SimpleTokenizer(max_vocab_size=3000)
tokenizer.build_vocab(texts)
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Test encode/decode
test_text = texts[0].split(".")[0].strip()  # First sentence
encoded = tokenizer.encode(test_text, max_len=32)
decoded = tokenizer.decode(encoded)
print(f"Original:  {test_text}")
print(f"Encoded:   {encoded[:15]}...")
print(f"Decoded:   {decoded}")

In [ ]:
# Tokenize all training data
SEQ_LEN = 32
train_ids = []
for text in texts:
    # Take first sentence only to keep sequences short
    sentence = text.split(".")[0].strip()
    if len(sentence.split()) >= 4:  # Skip very short sentences
        ids = tokenizer.encode(sentence, max_len=SEQ_LEN)
        train_ids.append(ids)

train_tensor = torch.tensor(train_ids, device=device)
print(f"Training data shape: {train_tensor.shape}")

## Task 2: Implement the Velocity Network

We provide the network architecture. You need to implement the flow matching training step.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.d_model // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float) / half
        )
        args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ff(self.norm2(x))
        return x


class VelocityNet(nn.Module):
    """Transformer velocity predictor for text embeddings."""

    def __init__(self, embed_dim=128, d_model=256, n_heads=4, n_layers=4, d_ff=512, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(embed_dim, d_model)
        self.time_embed = SinusoidalTimeEmbedding(d_model)
        self.time_proj = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model),
        )
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.output_norm = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, embed_dim)

    def forward(self, x_t, t):
        h = self.input_proj(x_t)
        t_emb = self.time_proj(self.time_embed(t))
        h = h + t_emb.unsqueeze(1)
        for block in self.blocks:
            h = block(h)
        h = self.output_norm(h)
        return self.output_proj(h)

In [ ]:
class FlowMatchingTextGenerator:
    """Flow matching text generator.

    Implements the full pipeline:
        tokens -> embeddings -> flow matching -> ODE sampling -> rounding -> tokens
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        d_model: int = 256,
        n_heads: int = 4,
        n_layers: int = 4,
        d_ff: int = 512,
        seq_len: int = 32,
        lr: float = 1e-4,
    ):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.seq_len = seq_len

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        nn.init.normal_(self.embedding.weight, std=0.02)

        self.velocity_net = VelocityNet(
            embed_dim=embed_dim, d_model=d_model, n_heads=n_heads,
            n_layers=n_layers, d_ff=d_ff,
        )

        self.optimizer = torch.optim.AdamW(
            list(self.embedding.parameters()) + list(self.velocity_net.parameters()),
            lr=lr,
        )

    def to(self, device):
        self.embedding.to(device)
        self.velocity_net.to(device)
        return self

    @property
    def device(self):
        return self.embedding.weight.device

    def train_step(self, token_ids: torch.Tensor) -> float:
        """One flow matching training step.

        Args:
            token_ids: (batch, seq_len) integer tensor.

        Returns:
            Scalar loss value.
        """
        self.velocity_net.train()
        batch_size = token_ids.shape[0]

        # TODO: Implement flow matching training step
        # 1. Embed tokens to get x_1 (target): x_1 = self.embedding(token_ids)
        # 2. Sample noise: x_0 = torch.randn_like(x_1)
        # 3. Sample time: t = torch.rand(batch_size, device=self.device)
        # 4. Interpolate: x_t = (1 - t) * x_0 + t * x_1
        #    (remember to expand t for broadcasting: t.unsqueeze(-1).unsqueeze(-1))
        # 5. Target velocity: v_target = x_1 - x_0
        # 6. Predict velocity: v_pred = self.velocity_net(x_t, t)
        # 7. Compute loss: F.mse_loss(v_pred, v_target)
        # 8. Backprop and step optimizer
        # 9. Return loss.item()
        pass

    @torch.no_grad()
    def sample(self, batch_size: int, n_steps: int = 100) -> torch.Tensor:
        """Generate token IDs via ODE sampling.

        Args:
            batch_size: Number of sequences to generate.
            n_steps: Number of Euler steps.

        Returns:
            (batch_size, seq_len) integer tensor of token IDs.
        """
        self.velocity_net.eval()

        # TODO: Implement ODE sampling
        # 1. Start from noise: x = torch.randn(batch_size, self.seq_len, self.embed_dim, device=self.device)
        # 2. Compute dt = 1.0 / n_steps
        # 3. Loop for n_steps:
        #    a. t = torch.full((batch_size,), step * dt, device=self.device)
        #    b. v = self.velocity_net(x, t)
        #    c. x = x + dt * v
        # 4. Round to tokens using cosine similarity:
        #    a. Normalize x and embedding table
        #    b. Compute similarity matrix
        #    c. Argmax to get token IDs
        # 5. Return token IDs
        pass

## Task 3: Train the Flow Matching Generator

In [ ]:
# Create and train the flow matching model
fm_gen = FlowMatchingTextGenerator(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128,
    d_model=256,
    n_heads=4,
    n_layers=4,
    d_ff=512,
    seq_len=SEQ_LEN,
    lr=3e-4,
)
fm_gen.to(device)

total_params = sum(p.numel() for p in fm_gen.velocity_net.parameters())
print(f"Velocity net parameters: {total_params:,}")

In [ ]:
# Training loop
BATCH_SIZE = 64
NUM_STEPS = 1000  # Increase for better quality (2000-5000 for real training)

fm_losses = []
for step in range(NUM_STEPS):
    idx = torch.randint(0, len(train_tensor), (BATCH_SIZE,))
    batch = train_tensor[idx]
    loss = fm_gen.train_step(batch)
    fm_losses.append(loss)

    if (step + 1) % 200 == 0:
        print(f"Step {step+1}/{NUM_STEPS}: loss = {loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(fm_losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Flow Matching Training Loss")
plt.show()

In [ ]:
# Generate samples
print("Generated text (Flow Matching, 50 ODE steps):")
tokens = fm_gen.sample(batch_size=8, n_steps=50)
for i in range(8):
    text = tokenizer.decode(tokens[i].cpu().tolist())
    print(f"  {i}: {text}")

## Task 4: Implement SDE Baseline

Implement a simplified DDPM-style text generator for comparison.

In [ ]:
class SDETextGenerator:
    """Simplified DDPM-style text generator (SDE baseline).

    Uses the same architecture as FlowMatchingTextGenerator but with:
    - DDPM-style forward noise addition
    - Stochastic reverse sampling
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        d_model: int = 256,
        n_heads: int = 4,
        n_layers: int = 4,
        d_ff: int = 512,
        seq_len: int = 32,
        num_timesteps: int = 1000,
        lr: float = 1e-4,
    ):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.seq_len = seq_len
        self.num_timesteps = num_timesteps

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        nn.init.normal_(self.embedding.weight, std=0.02)

        # Reuse same architecture but for noise prediction
        self.noise_net = VelocityNet(
            embed_dim=embed_dim, d_model=d_model, n_heads=n_heads,
            n_layers=n_layers, d_ff=d_ff,
        )

        # Noise schedule
        betas = torch.linspace(1e-4, 0.02, num_timesteps)
        alphas = 1.0 - betas
        self.alpha_bars = torch.cumprod(alphas, dim=0)
        self.betas = betas
        self.alphas = alphas

        self.optimizer = torch.optim.AdamW(
            list(self.embedding.parameters()) + list(self.noise_net.parameters()),
            lr=lr,
        )

    def to(self, device):
        self.embedding.to(device)
        self.noise_net.to(device)
        self.alpha_bars = self.alpha_bars.to(device)
        self.betas = self.betas.to(device)
        self.alphas = self.alphas.to(device)
        return self

    @property
    def device(self):
        return self.embedding.weight.device

    def train_step(self, token_ids: torch.Tensor) -> float:
        """One DDPM training step (noise prediction)."""
        self.noise_net.train()
        batch_size = token_ids.shape[0]

        # TODO: Implement DDPM training step
        # 1. Embed tokens to get x_0
        # 2. Sample timesteps t ~ Uniform{0, ..., T-1}
        # 3. Sample noise epsilon ~ N(0, I)
        # 4. Compute x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon
        # 5. Predict noise: epsilon_pred = self.noise_net(x_t, t.float() / self.num_timesteps)
        #    (normalize t to [0, 1] for the time embedding)
        # 6. Loss = MSE(epsilon_pred, epsilon)
        # 7. Backprop and step
        pass

    @torch.no_grad()
    def sample(self, batch_size: int, n_steps: int = None) -> torch.Tensor:
        """Generate token IDs via DDPM reverse process.

        Args:
            batch_size: Number of sequences.
            n_steps: If None, use all num_timesteps. Otherwise subsample.

        Returns:
            (batch_size, seq_len) token IDs.
        """
        self.noise_net.eval()
        if n_steps is None:
            n_steps = self.num_timesteps

        # TODO: Implement DDPM sampling
        # 1. Start from noise: x = torch.randn(...)
        # 2. Create timestep schedule (evenly spaced from T-1 to 0)
        # 3. For each timestep t (going backwards):
        #    a. Predict noise: eps = self.noise_net(x, t_normalized)
        #    b. Compute x_0 estimate: x_0_hat = (x - sqrt(1-alpha_bar_t) * eps) / sqrt(alpha_bar_t)
        #    c. Update x using DDPM update rule (add noise if not last step)
        # 4. Round to tokens
        pass

In [ ]:
# Create and train SDE baseline
sde_gen = SDETextGenerator(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128,
    d_model=256,
    n_heads=4,
    n_layers=4,
    d_ff=512,
    seq_len=SEQ_LEN,
    num_timesteps=1000,
    lr=3e-4,
)
sde_gen.to(device)

# Train for the same number of steps
sde_losses = []
for step in range(NUM_STEPS):
    idx = torch.randint(0, len(train_tensor), (BATCH_SIZE,))
    batch = train_tensor[idx]
    loss = sde_gen.train_step(batch)
    sde_losses.append(loss)

    if (step + 1) % 200 == 0:
        print(f"Step {step+1}/{NUM_STEPS}: loss = {loss:.4f}")

# Plot both losses
plt.figure(figsize=(8, 4))
plt.plot(fm_losses, label="Flow Matching (ODE)", alpha=0.7)
plt.plot(sde_losses, label="DDPM (SDE)", alpha=0.7)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.title("Training Loss Comparison")
plt.show()

## Task 5: Compare ODE vs SDE Generation

In [ ]:
# TODO: Compare generation speed and quality

# 1. Generate with flow matching at different step counts
fm_step_counts = [10, 25, 50, 100]
print("=== Flow Matching (ODE) ===")
for n_steps in fm_step_counts:
    start = time.time()
    tokens = fm_gen.sample(batch_size=16, n_steps=n_steps)
    elapsed = (time.time() - start) * 1000
    nfe = n_steps  # One model forward pass per step
    sample_text = tokenizer.decode(tokens[0].cpu().tolist())
    print(f"  Steps={n_steps:4d} | NFE={nfe:4d} | Time={elapsed:7.1f}ms | Sample: {sample_text}")

# 2. Generate with SDE at different step counts
sde_step_counts = [50, 100, 250, 500, 1000]
print("\n=== DDPM (SDE) ===")
for n_steps in sde_step_counts:
    start = time.time()
    tokens = sde_gen.sample(batch_size=16, n_steps=n_steps)
    elapsed = (time.time() - start) * 1000
    nfe = n_steps
    sample_text = tokenizer.decode(tokens[0].cpu().tolist())
    print(f"  Steps={n_steps:4d} | NFE={nfe:4d} | Time={elapsed:7.1f}ms | Sample: {sample_text}")

In [ ]:
# TODO: Create a comparison plot
# Plot wall-clock time vs NFE for both methods
# You should observe that flow matching achieves comparable quality with fewer steps

# Collect timing data
fm_times = []
sde_times = []

for n_steps in fm_step_counts:
    start = time.time()
    _ = fm_gen.sample(batch_size=16, n_steps=n_steps)
    fm_times.append((time.time() - start) * 1000)

for n_steps in sde_step_counts:
    start = time.time()
    _ = sde_gen.sample(batch_size=16, n_steps=n_steps)
    sde_times.append((time.time() - start) * 1000)

plt.figure(figsize=(8, 5))
plt.plot(fm_step_counts, fm_times, 'o-', label='Flow Matching (ODE)', linewidth=2)
plt.plot(sde_step_counts, sde_times, 's-', label='DDPM (SDE)', linewidth=2)
plt.xlabel('Number of Function Evaluations (NFE)')
plt.ylabel('Wall-clock Time (ms)')
plt.title('Generation Speed: ODE vs SDE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Analysis Questions

After completing the tasks above, answer these questions:

1. At how many ODE steps does flow matching produce reasonable text? How does this compare to the SDE?
2. Is there a quality difference between ODE and SDE generation at their respective best settings?
3. What is the speedup ratio (SDE time / ODE time) for comparable quality?
4. Why does flow matching need fewer steps? (Hint: think about the path geometry.)
5. What are the downsides of deterministic (ODE) generation compared to stochastic (SDE)?